In [1]:
# ============================================================
# PharmaLens AI
# Data Pipeline
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Project Path
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

RAW_DATA_PATH = PROJECT_ROOT / "IMS (2021-2025).xlsx"


# ------------------------------------------------------------
# Expected Columns
# ------------------------------------------------------------

EXPECTED_COLUMNS = [
    "Distribution Channel",
    "Therapeutic Class",
    "Manufacturer",
    "Brand Name",
    "Pack Size",
    "Product Launch",
    "Drug Strength",
    "Selling Price",
    "Market Category",
    "Month",
    "Year",
    "Sales Units",
    "Sales Value"
]


# ------------------------------------------------------------
# Load Raw Data
# ------------------------------------------------------------

def load_raw_data():

    df = pd.read_excel(RAW_DATA_PATH)

    print("Raw dataset loaded successfully!")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")

    return df


# ------------------------------------------------------------
# Validate Columns
# ------------------------------------------------------------

def validate_columns(df):

    missing_columns = [
        col for col in EXPECTED_COLUMNS
        if col not in df.columns
    ]

    if missing_columns:

        raise ValueError(
            f"Missing columns: {missing_columns}"
        )

    print("Column validation passed!")

    return True


# ------------------------------------------------------------
# Clean Data
# ------------------------------------------------------------

def clean_data(df):

    df = df.copy()

    # Remove duplicate rows
    df = df.drop_duplicates()

    # Text columns
    text_columns = [
        "Distribution Channel",
        "Therapeutic Class",
        "Manufacturer",
        "Brand Name",
        "Pack Size",
        "Drug Strength",
        "Market Category"
    ]

    for col in text_columns:

        if col in df.columns:

            df[col] = (
                df[col]
                .astype("string")
                .str.strip()
            )

    # Date columns
    if "Product Launch" in df.columns:

        df["Product Launch"] = pd.to_datetime(
            df["Product Launch"],
            errors="coerce"
        )

    if "Month" in df.columns:

        df["Month"] = pd.to_datetime(
            df["Month"],
            errors="coerce"
        )

    # Numeric columns
    numeric_columns = [
        "Selling Price",
        "Sales Units",
        "Sales Value",
        "Year"
    ]

    for col in numeric_columns:

        if col in df.columns:

            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    # Remove impossible values
    if "Sales Units" in df.columns:

        df.loc[
            df["Sales Units"] < 0,
            "Sales Units"
        ] = np.nan

    if "Sales Value" in df.columns:

        df.loc[
            df["Sales Value"] < 0,
            "Sales Value"
        ] = np.nan

    if "Selling Price" in df.columns:

        df.loc[
            df["Selling Price"] < 0,
            "Selling Price"
        ] = np.nan

    return df


# ------------------------------------------------------------
# Feature Engineering
# ------------------------------------------------------------

def create_master_features(df):

    df = df.copy()

    # Revenue per unit
    if (
        "Sales Value" in df.columns
        and "Sales Units" in df.columns
    ):

        df["Revenue_per_Unit"] = np.where(
            df["Sales Units"] > 0,
            df["Sales Value"] / df["Sales Units"],
            np.nan
        )

    # Launch year
    if "Product Launch" in df.columns:

        df["Launch_Year"] = (
            df["Product Launch"].dt.year
        )

    # Product age
    if (
        "Year" in df.columns
        and "Launch_Year" in df.columns
    ):

        df["Product_Age_Years"] = (
            df["Year"] - df["Launch_Year"]
        )

    # New product flag
    if "Product_Age_Years" in df.columns:

        df["New_Product_Flag"] = np.where(
            df["Product_Age_Years"] <= 2,
            1,
            0
        )

    # Month number
    if "Month" in df.columns:

        df["Month_Number"] = (
            df["Month"].dt.month
        )

    return df


# ------------------------------------------------------------
# Build Master Dataset
# ------------------------------------------------------------

def build_master_dataset():

    df = load_raw_data()

    validate_columns(df)

    df = clean_data(df)

    df = create_master_features(df)

    print("\nMaster dataset created successfully!")

    print(
        f"Final shape: {df.shape}"
    )

    return df


# ------------------------------------------------------------
# Main Function
# ------------------------------------------------------------

def get_master_data():

    return build_master_dataset()